# 🍕 Redis Eats RAG Workshop
## *Don't Talk With Food In Your Mouth*

Welcome! In this workshop you will build a **Retrieval-Augmented Generation (RAG) chatbot** for Redis Eats — a food delivery platform — using:

| Component | Role |
|---|---|
| **Redis Cloud** | Vector database, semantic cache, and router store |
| **RedisVL** | Python library for Redis AI application patterns |
| **Redis Query Engine** | High-performance vector similarity search |
| **Redis LangCache** | Semantic caching for LLM responses |
| **OpenAI** | Embeddings (`text-embedding-3-small`) and chat completions |

---

## What You Will Build

A working support chatbot called **Don't Talk With Food In Your Mouth** that:

- Reads food-delivery **policy PDFs** and stores them in Redis as vectors
- Answers questions by **retrieving relevant chunks** and calling an LLM
- Returns **source citations** alongside every answer
- **Refuses off-topic questions** before they ever hit the database
- **Caches repeated questions** to avoid unnecessary LLM calls

---

## Architecture

![Architecture](../docs/architecture/redis-eats-rag-workshop-architecture.png)

---

## What This Workshop Covers (Workshop 1)

- Redis Cloud connection via RedisVL
- PDF loading and chunking
- OpenAI embeddings
- RedisVL `SearchIndex` + Redis Query Engine vector search
- RAG answer generation with citations
- Semantic routing with RedisVL `SemanticRouter`
- Semantic caching with Redis LangCache

## What Is Saved for Workshop 2

Conversational memory, agentic workflows, MCP tools, Redis Iris, live order lookup, and context-aware responses.

---

**Estimated time:** ~2 hours  
**Difficulty:** Beginner–Intermediate

---
## 📋 Table of Contents

| # | Section |
|---|---|
| 0 | Welcome and What We Are Building |
| 1 | Setup — packages, credentials, connectivity tests |
| 2 | Meet the Data — load and preview policy PDFs |
| 3 | Chunking — split documents for retrieval |
| 4 | Embeddings — convert text to vectors |
| 5 | RedisVL Schema and Index Creation |
| 6 | Load Chunks into Redis |
| 7 | Vector Search |
| 8 | Build the RAG Answer Function |
| 9 | Add Semantic Routing |
| 10 | Add LangCache |
| 11 | Final Chatbot Test |
| 12 | Reset Lab |
| 13 | What Comes Next |


In [ ]:
#@title ⚙️ Google Colab Setup (run this first if using Colab) { display-mode: 'form' }
# ---------------------------------------------------------------------------
# If you are running in Google Colab, run this cell first to clone the repo
# so the data/pdfs/ folder is available.
#
# If you are running locally, skip this cell.
# ---------------------------------------------------------------------------
import os
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('/content/redis-eats-rag-workshop'):
        print('Cloning workshop repo...')
        os.system('git clone https://github.com/testuser/redis-eats-rag-workshop /content/redis-eats-rag-workshop')
        os.chdir('/content/redis-eats-rag-workshop')
        print('Done.')
    else:
        os.chdir('/content/redis-eats-rag-workshop')
        print('Repo already present.')
else:
    print('Not running in Colab — skipping repo clone.')

---
## Section 1 — Setup

First, install the required Python packages. This cell is safe to re-run.


In [ ]:
# Install required packages
# This works in Google Colab and local Jupyter environments.
%pip install redis redisvl openai pypdf langcache tqdm python-dotenv --quiet

In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
import os
import uuid
import json
import getpass
from pathlib import Path
from typing import List, Dict, Any

import redis as redis_lib
from redisvl.index import SearchIndex
from redisvl.schema import IndexSchema
from redisvl.query import VectorQuery

import openai
from openai import OpenAI

import pypdf
from tqdm import tqdm

print("✅ Imports complete")

### 1.1 — Credentials

Enter your **Redis Cloud** and **OpenAI** credentials below. They are stored only in this notebook session — nothing is written to disk.

> **Redis Cloud:** You need the host, port, and password for your database.  
> **OpenAI:** A paid API key starting with `sk-`.


In [ ]:
# ---------------------------------------------------------------------------
# Collect credentials interactively
# getpass hides sensitive values so they don't appear in notebook output.
# ---------------------------------------------------------------------------

REDIS_HOST = input("Redis Cloud host (e.g. redis-12345.c1.us-east-1-2.ec2.redns.redis-cloud.com): ").strip()
REDIS_PORT = int(input("Redis port [6379]: ").strip() or "6379")
REDIS_USERNAME = input("Redis username [default]: ").strip() or "default"
REDIS_PASSWORD = getpass.getpass("Redis password: ")

OPENAI_API_KEY = getpass.getpass("OpenAI API key: ")

# Build the Redis URL used by RedisVL throughout this workshop
REDIS_URL = f"rediss://{REDIS_USERNAME}:{REDIS_PASSWORD}@{REDIS_HOST}:{REDIS_PORT}"

print("✅ Credentials collected")

### 1.2 — Test Redis Connectivity

RedisVL will connect to Redis Cloud over TLS (`rediss://`). Let's verify the connection before doing any real work.


In [ ]:
# ---------------------------------------------------------------------------
# Section 1.2 — Redis Cloud Connectivity Check
#
# This cell runs four checks and must pass before you continue:
#   1. TCP + TLS connection to Redis Cloud
#   2. Redis server version (must be 7.x+ for Redis Query Engine)
#   3. Redis Search module loaded (required for FT.CREATE and vector search)
#   4. RedisVL can use the REDIS_URL (used for SearchIndex throughout the notebook)
# ---------------------------------------------------------------------------

import sys

_redis_ok = True   # Set to False on any failure so later cells can gate on this

# --- Check 1: TCP + TLS connection ---
print("Checking Redis Cloud connectivity...\n")

try:
    r = redis_lib.Redis(
        host=REDIS_HOST,
        port=REDIS_PORT,
        username=REDIS_USERNAME,
        password=REDIS_PASSWORD,
        ssl=True,                  # Redis Cloud always requires TLS
        decode_responses=True,
        socket_connect_timeout=10,
        socket_timeout=10,
    )
    r.ping()
    print("  ✅ TCP + TLS connection  OK")
except redis_lib.exceptions.AuthenticationError:
    print("  ❌ Authentication failed")
    print("     → Check your Redis username and password in the credentials cell above")
    _redis_ok = False
except redis_lib.exceptions.ConnectionError as e:
    err = str(e).lower()
    if "ssl" in err or "tls" in err or "certificate" in err:
        print(f"  ❌ TLS/SSL error: {e}")
        print("     → Make sure ssl=True is set. Redis Cloud requires rediss:// (TLS)")
    elif "timed out" in err or "timeout" in err:
        print(f"  ❌ Connection timed out")
        print("     → Check the host and port. Is the database in Active state?")
    else:
        print(f"  ❌ Connection error: {e}")
        print("     → Verify REDIS_HOST and REDIS_PORT are correct")
    _redis_ok = False
except Exception as e:
    print(f"  ❌ Unexpected error: {e}")
    _redis_ok = False

# --- Check 2: Redis server version ---
if _redis_ok:
    try:
        info = r.info("server")
        version = info.get("redis_version", "unknown")
        major   = int(version.split(".")[0]) if version != "unknown" else 0
        if major >= 7:
            print(f"  ✅ Redis version         {version}  (Redis Query Engine supported)")
        else:
            print(f"  ⚠️  Redis version         {version}  (recommend 7.x+ for this workshop)")
    except Exception as e:
        print(f"  ⚠️  Could not read server version: {e}")

# --- Check 3: Redis Search module ---
if _redis_ok:
    try:
        modules = r.execute_command("MODULE LIST")
        module_names = []
        for m in modules:
            # MODULE LIST returns a list of lists: [[b'name', b'search', ...], ...]
            if isinstance(m, list):
                for i, item in enumerate(m):
                    if item in (b"name", "name") and i + 1 < len(m):
                        module_names.append(str(m[i + 1]).lower())
        search_loaded = any("search" in n for n in module_names)
        if search_loaded:
            print(f"  ✅ Redis Search module   loaded  (FT.CREATE and vector search ready)")
        else:
            # Fallback: try FT._LIST directly — some Redis builds expose search without MODULE LIST
            try:
                r.execute_command("FT._LIST")
                print(f"  ✅ Redis Search module   available  (FT._LIST succeeded)")
            except Exception:
                print(f"  ❌ Redis Search module   NOT found")
                print(f"     → This workshop requires Redis Stack or Redis Cloud with Search enabled")
                print(f"     → Upgrade your Redis Cloud database to a plan that includes Search")
                _redis_ok = False
    except Exception:
        # Some managed Redis endpoints block MODULE LIST — try FT._LIST as fallback
        try:
            r.execute_command("FT._LIST")
            print(f"  ✅ Redis Search module   available  (FT._LIST succeeded)")
        except Exception as e2:
            print(f"  ❌ Redis Search module check failed: {e2}")
            print(f"     → Redis Cloud free tier includes Search. Re-check your database plan.")
            _redis_ok = False

# --- Check 4: RedisVL can use REDIS_URL ---
if _redis_ok:
    try:
        from redisvl.redis.connection import RedisConnectionFactory
        test_client = RedisConnectionFactory.get_redis_connection(url=REDIS_URL)
        test_client.ping()
        print(f"  ✅ RedisVL connection     OK  (REDIS_URL is valid for SearchIndex)")
    except Exception as e:
        print(f"  ❌ RedisVL connection failed: {e}")
        print(f"     → REDIS_URL = {REDIS_URL[:40]}...")
        print(f"     → Check that host, port, username, and password are all correct")
        _redis_ok = False

# --- Summary ---
print()
if _redis_ok:
    print("✅ All Redis checks passed — ready to continue")
    print(f"   Host    : {REDIS_HOST}:{REDIS_PORT}")
    print(f"   User    : {REDIS_USERNAME}")
else:
    print("❌ Redis setup has issues — fix the errors above before continuing")
    print("   Do NOT run the rest of the notebook until all checks pass.")


### 1.3 — Test OpenAI Connectivity


In [ ]:
# ---------------------------------------------------------------------------
# Section 1.3 — OpenAI Connectivity Check
#
# This cell runs three checks and must pass before you continue:
#   1. API key is valid (authentication)
#   2. Embedding model works + returns correct dimensions (catches quota issues)
#   3. Chat model responds (catches quota issues on the chat endpoint)
#
# A key that passes auth but has $0 credits will fail checks 2 and 3 —
# catching that early avoids a surprise failure during the batch embed step.
# ---------------------------------------------------------------------------

_openai_ok = True   # Gate flag used by later cells

print("Checking OpenAI connectivity...\n")

# --- Check 1: Create client + authenticate ---
try:
    openai_client = OpenAI(api_key=OPENAI_API_KEY)
    # A lightweight authenticated call — doesn't consume credits
    openai_client.models.list()
    print("  ✅ API key               valid")
except openai.AuthenticationError:
    print("  ❌ API key invalid")
    print("     → Check your key starts with sk- and hasn't been revoked")
    _openai_ok = False
except openai.PermissionDeniedError:
    print("  ❌ API key does not have permission")
    print("     → Make sure this is a paid OpenAI account key")
    _openai_ok = False
except Exception as e:
    print(f"  ❌ OpenAI connection error: {e}")
    _openai_ok = False

# --- Check 2: Embedding model ---
if _openai_ok:
    try:
        test_resp = openai_client.embeddings.create(
            model="text-embedding-3-small",
            input="connectivity test",
        )
        dims = len(test_resp.data[0].embedding)
        if dims == 1536:
            print(f"  ✅ Embedding model       text-embedding-3-small  ({dims} dims)")
        else:
            print(f"  ⚠️  Embedding model       returned {dims} dims (expected 1536)")
            print(f"     → The RedisVL schema is set to 1536 dims. Check the model name.")
    except openai.RateLimitError:
        print("  ❌ Embedding model       rate limit hit or insufficient credits")
        print("     → Check your OpenAI account billing at platform.openai.com")
        _openai_ok = False
    except openai.NotFoundError:
        print("  ❌ Embedding model       text-embedding-3-small not accessible")
        print("     → Verify your account has access to this model")
        _openai_ok = False
    except Exception as e:
        print(f"  ❌ Embedding model error: {e}")
        _openai_ok = False

# --- Check 3: Chat model ---
if _openai_ok:
    try:
        chat_resp = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": "Reply with the single word: ready"}],
            max_tokens=5,
            temperature=0,
        )
        reply = chat_resp.choices[0].message.content.strip().lower()
        print(f"  ✅ Chat model            gpt-4o-mini  (test response: '{reply}')")
    except openai.RateLimitError:
        print("  ❌ Chat model            rate limit hit or insufficient credits")
        print("     → Check your OpenAI account billing at platform.openai.com")
        _openai_ok = False
    except openai.NotFoundError:
        print("  ❌ Chat model            gpt-4o-mini not accessible")
        print("     → Verify your account has access to this model")
        _openai_ok = False
    except Exception as e:
        print(f"  ❌ Chat model error: {e}")
        _openai_ok = False

# --- Summary ---
print()
if _openai_ok:
    print("✅ All OpenAI checks passed — ready to continue")
else:
    print("❌ OpenAI setup has issues — fix the errors above before continuing")
    print("   Common fixes:")
    print("   • Ensure your OpenAI account has a credit card and available balance")
    print("   • Check usage limits at: https://platform.openai.com/usage")
    print("   • Make sure the key was not copy/pasted with extra whitespace")


---
## Section 2 — Meet the Data

The bot will answer questions from **10 Redis Eats policy and procedure documents**. These PDFs cover refunds, cancellations, delivery delays, driver procedures, restaurant onboarding, food safety, promo codes, account help, order status, and customer support.

Let's load and inspect them.


In [ ]:
# ---------------------------------------------------------------------------
# Locate PDF files
# Supports running from Google Colab (repo cloned) or locally.
# ---------------------------------------------------------------------------

# Try paths relative to common notebook launch locations
CANDIDATE_DIRS = [
    Path("../data/pdfs"),       # local: launched from notebooks/
    Path("data/pdfs"),          # local: launched from repo root
    Path("/content/redis-eats-rag-workshop/data/pdfs"),  # Colab
]

PDF_DIR = None
for candidate in CANDIDATE_DIRS:
    if candidate.exists() and list(candidate.glob("*.pdf")):
        PDF_DIR = candidate
        break

if PDF_DIR is None:
    raise FileNotFoundError(
        "Could not find data/pdfs/. "
        "Clone the repo and make sure the PDFs are present, "
        "or run scripts/generate_policy_pdfs.py to generate them."
    )

pdf_files = sorted(PDF_DIR.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDFs in {PDF_DIR}\n")
for f in pdf_files:
    print(f"  📄 {f.name}")

### 2.1 — Preview a Document

Let's look at the first page of the refund policy so we know what the bot will be answering from.


In [ ]:
# ---------------------------------------------------------------------------
# Preview the first page of one policy document
# ---------------------------------------------------------------------------
preview_path = PDF_DIR / "refund_policy.pdf"

with open(preview_path, "rb") as f:
    reader = pypdf.PdfReader(f)
    first_page_text = reader.pages[0].extract_text()

print(f"--- {preview_path.name} (page 1) ---\n")
print(first_page_text[:1500])  # Show first 1500 characters

---
## Section 3 — Chunking

### Why Do We Chunk?

When we store documents in a vector database, we store them as **chunks** — small overlapping pieces of text rather than entire pages or files. This matters for two reasons:

1. **Retrieval precision.** Embedding models compress text into a fixed-size vector. A short, focused chunk produces a more accurate embedding than a long, multi-topic page.
2. **LLM context limits.** We pass retrieved text directly into a prompt. Smaller chunks let us pack more relevant content without hitting token limits.

### Chunk Parameters

| Parameter | What It Does |
|---|---|
| `chunk_size` | Maximum number of characters per chunk |
| `chunk_overlap` | Characters shared between adjacent chunks (preserves context at boundaries) |

We'll use **character-level chunking** here — simple and easy to reason about.


In [ ]:
# ---------------------------------------------------------------------------
# Extract all text from a PDF file, page by page
# ---------------------------------------------------------------------------
def extract_text_from_pdf(pdf_path: Path) -> List[Dict[str, Any]]:
    """
    Extract text from every page of a PDF.

    Returns a list of dicts, one per page, each containing:
      - text        : the extracted page text
      - source      : PDF filename (without path)
      - page_number : 1-based page number
    """
    pages = []
    with open(pdf_path, "rb") as f:
        reader = pypdf.PdfReader(f)
        for page_num, page in enumerate(reader.pages, start=1):
            text = page.extract_text() or ""
            text = text.strip()
            if text:  # Skip blank pages
                pages.append({
                    "text": text,
                    "source": pdf_path.name,
                    "page_number": page_num,
                })
    return pages


# Test on one file
sample_pages = extract_text_from_pdf(PDF_DIR / "refund_policy.pdf")
print(f"refund_policy.pdf → {len(sample_pages)} page(s) extracted")

In [ ]:
# ---------------------------------------------------------------------------
# Chunk a single page of text into overlapping segments
# ---------------------------------------------------------------------------
def chunk_text(
    text: str,
    source: str,
    page_number: int,
    chunk_size: int = 500,
    chunk_overlap: int = 50,
) -> List[Dict[str, Any]]:
    """
    Split text into overlapping character-level chunks.

    Each chunk dict contains:
      - chunk_id    : unique identifier for this chunk
      - text        : the chunk text
      - source      : source PDF filename
      - page_number : page the chunk came from
      - chunk_index : position of this chunk within the source page
    """
    chunks = []
    start = 0
    chunk_index = 0

    while start < len(text):
        end = start + chunk_size
        chunk_text_str = text[start:end]

        chunks.append({
            "chunk_id": str(uuid.uuid4()),
            "text": chunk_text_str,
            "source": source,
            "page_number": page_number,
            "chunk_index": chunk_index,
        })

        # Advance by chunk_size minus overlap so adjacent chunks share context
        start += chunk_size - chunk_overlap
        chunk_index += 1

    return chunks


# Quick test: chunk the first page of the refund policy
sample_chunks = chunk_text(
    text=sample_pages[0]["text"],
    source=sample_pages[0]["source"],
    page_number=sample_pages[0]["page_number"],
)
print(f"First page split into {len(sample_chunks)} chunk(s)")
print(f"\nChunk 0 ({len(sample_chunks[0]['text'])} chars):")
print(sample_chunks[0]["text"][:300], "...")

### 🏋️ Exercise 3.1 — Adjust Chunk Settings

The default settings are `chunk_size=500, chunk_overlap=50`. Try changing them and observe the effect on chunk count and content.

**Fill in the values below** and re-run the cell:

- What happens if you make `chunk_size` much smaller (e.g. 200)?
- What happens if you set `chunk_overlap` to 0?
- What value feels right for a few sentences of policy text?

> 💡 **Hint:** Good chunk sizes for RAG over short policy documents are usually 300–800 characters with 50–100 characters of overlap.


In [ ]:
# ---------------------------------------------------------------------------
# Exercise: Fill in YOUR chunk settings and observe the result
# ---------------------------------------------------------------------------

MY_CHUNK_SIZE    = 500   # TODO: change this and re-run
MY_CHUNK_OVERLAP = 50    # TODO: change this and re-run

exercise_chunks = chunk_text(
    text=sample_pages[0]["text"],
    source=sample_pages[0]["source"],
    page_number=sample_pages[0]["page_number"],
    chunk_size=MY_CHUNK_SIZE,
    chunk_overlap=MY_CHUNK_OVERLAP,
)

print(f"chunk_size={MY_CHUNK_SIZE}, chunk_overlap={MY_CHUNK_OVERLAP}")
print(f"→ {len(exercise_chunks)} chunks from first page")
for i, c in enumerate(exercise_chunks[:3]):
    print(f"  Chunk {i}: {len(c['text'])} chars — {c['text'][:80]!r}...")

In [ ]:
#@title ✅ Solution — Recommended Chunk Settings { display-mode: 'form' }

# Recommended values for this workshop:
#
#   chunk_size=500    — captures a full policy paragraph in most cases
#   chunk_overlap=50  — preserves a sentence of context at chunk boundaries
#
# These are the defaults already used in chunk_text().
# For your own projects, experiment with 300–800 and measure retrieval quality.

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
print(f"Using chunk_size={CHUNK_SIZE}, chunk_overlap={CHUNK_OVERLAP}")

In [ ]:
# ---------------------------------------------------------------------------
# Process all 10 PDFs into chunks
# ---------------------------------------------------------------------------

# Use the recommended settings
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

all_chunks: List[Dict[str, Any]] = []

for pdf_path in tqdm(pdf_files, desc="Processing PDFs"):
    pages = extract_text_from_pdf(pdf_path)
    for page in pages:
        chunks = chunk_text(
            text=page["text"],
            source=page["source"],
            page_number=page["page_number"],
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
        )
        all_chunks.extend(chunks)

print(f"\n✅ Total chunks: {len(all_chunks)}")
print(f"   Average chunk length: {sum(len(c['text']) for c in all_chunks) // len(all_chunks)} chars")

# Show a sample chunk
sample = all_chunks[5]
print(f"\nSample chunk from '{sample['source']}' (page {sample['page_number']}):")
print(sample["text"])

---
## Section 4 — Embeddings

### What Is an Embedding?

An **embedding** is a list of numbers (a vector) that represents the *meaning* of a piece of text. Two pieces of text with similar meaning will produce vectors that are close together in vector space — even if they use completely different words.

This is what makes **semantic search** possible. Instead of matching exact keywords, Redis finds chunks whose *meaning* is closest to the user's question.

We'll use OpenAI's `text-embedding-3-small` model, which produces **1536-dimensional** vectors.

```
"Can I get a refund?"  →  [0.012, -0.043, 0.891, ...]  (1536 numbers)
"Refund policy"        →  [0.015, -0.039, 0.887, ...]  (very close!)
"Football scores"      →  [-0.234, 0.512, -0.103, ...]  (far away)
```

Redis stores these vectors alongside each chunk and finds the nearest ones at query time.


In [ ]:
# ---------------------------------------------------------------------------
# Embedding model configuration
# ---------------------------------------------------------------------------

EMBEDDING_MODEL = "text-embedding-3-small"  # OpenAI model
EMBEDDING_DIMS  = 1536                       # Dimensions produced by this model

def get_embedding(text: str) -> List[float]:
    """
    Generate a single embedding vector for the given text using OpenAI.

    Args:
        text: The input string to embed.

    Returns:
        A list of 1536 floats.
    """
    response = openai_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text,
    )
    return response.data[0].embedding


# Test: embed one sentence and check the output shape
test_vec = get_embedding("Can I get a refund if my food arrived cold?")
print(f"✅ Embedding generated")
print(f"   Dimensions : {len(test_vec)}")
print(f"   First 5 values: {[round(v, 4) for v in test_vec[:5]]}")

In [ ]:
# ---------------------------------------------------------------------------
# Generate embeddings for all chunks
# This will make one OpenAI API call per chunk.
# With ~100–200 chunks it typically takes under 60 seconds.
# ---------------------------------------------------------------------------

print(f"Generating embeddings for {len(all_chunks)} chunks...")

for chunk in tqdm(all_chunks, desc="Embedding"):
    chunk["embedding"] = get_embedding(chunk["text"])

print(f"\n✅ All embeddings generated")
print(f"   Each vector: {len(all_chunks[0]['embedding'])} dimensions")

---
## Section 5 — RedisVL Schema and Index Creation

### Redis as a Vector Database

If you've used Redis as a cache, you're probably familiar with simple key-value operations. Redis Cloud also includes the **Redis Query Engine**, which lets you define *indexes* over your data and run sophisticated queries — including **vector similarity search**.

An **index** tells Redis which fields to index and how to index them. Once the index is created, Redis maintains it automatically as data is written.

### RedisVL Schema

**RedisVL** is a Python library that makes it easy to define schemas, create indexes, load data, and run queries — all with clean, readable code.

Our schema has five fields:

| Field | Type | Purpose |
|---|---|---|
| `text` | `text` | The chunk content — full-text searchable |
| `source` | `tag` | PDF filename — filterable |
| `page_number` | `numeric` | Page number — filterable |
| `chunk_index` | `numeric` | Position within page |
| `embedding` | `vector` | 1536-dim COSINE vector for similarity search |

We use the **FLAT** algorithm — exact brute-force search, which is ideal for small datasets like this workshop where 100% recall matters and speed is not a bottleneck.

> For production datasets with millions of vectors, use **HNSW** (approximate nearest neighbor) for much faster queries. Redis supports both.


In [ ]:
# ---------------------------------------------------------------------------
# Define the RedisVL schema for our chunk index
#
# Key prefix: redis-eats:chunk:
#   Each chunk will be stored as a Redis Hash at a key like:
#   redis-eats:chunk:3f2a1b9c-...
#
# Vector algorithm: FLAT
#   Exact nearest-neighbor search — best for small datasets and workshops
#   where we want guaranteed accuracy with no approximation.
#
# Distance metric: COSINE
#   Measures the angle between vectors (ignores magnitude).
#   text-embedding-3-small produces cosine-compatible embeddings.
# ---------------------------------------------------------------------------

INDEX_NAME   = "redis-eats-chunks"       # Name of the Redis search index
KEY_PREFIX   = "redis-eats:chunk:"       # All chunk keys share this prefix

schema_dict = {
    "index": {
        "name": INDEX_NAME,
        "prefix": KEY_PREFIX,
        "storage_type": "hash",          # Store each chunk as a Redis Hash
    },
    "fields": [
        {"name": "text",        "type": "text"},
        {"name": "source",      "type": "tag"},
        {"name": "page_number", "type": "numeric"},
        {"name": "chunk_index", "type": "numeric"},
        {
            "name": "embedding",
            "type": "vector",
            "attrs": {
                "dims":            EMBEDDING_DIMS,
                "algorithm":       "FLAT",    # Exact search — good for ~100–1000 vectors
                "distance_metric": "COSINE",
                "datatype":        "FLOAT32",
            },
        },
    ],
}

schema = IndexSchema.from_dict(schema_dict)
print("✅ Schema defined")
print(f"   Index name : {INDEX_NAME}")
print(f"   Key prefix : {KEY_PREFIX}")
print(f"   Fields     : {[f.name for f in schema.fields.values()]}")

In [ ]:
# ---------------------------------------------------------------------------
# Create the RedisVL SearchIndex
#
# SearchIndex wraps the schema and provides load(), search(), and delete()
# methods. We pass the Redis URL so RedisVL manages the connection pool.
# ---------------------------------------------------------------------------

index = SearchIndex(schema, redis_url=REDIS_URL)

# Create the index on Redis Cloud.
# overwrite=True means if the index already exists from a previous run,
# it will be dropped and recreated cleanly.
index.create(overwrite=True)

print(f"✅ Index '{INDEX_NAME}' created on Redis Cloud")

# Verify the index exists by fetching its info
info = index.info()
print(f"   Num docs indexed : {info.get('num_docs', 0)}")
print(f"   Index state      : {info.get('index_definition', {}).get('key_type', 'HASH')}")

---
## Section 6 — Load Chunks into Redis

Now we write all our chunks — text, metadata, and embedding vectors — into Redis Cloud. RedisVL's `index.load()` handles batching and serialization for us.

Each chunk is stored as a **Redis Hash** at a key like `redis-eats:chunk:<uuid>`. The embedding vector is stored as a binary field alongside the text and metadata. Redis maintains the vector index automatically as data is written.


In [ ]:
# ---------------------------------------------------------------------------
# Prepare records for RedisVL
#
# Each record is a flat dict matching the schema fields.
# The chunk_id becomes the unique part of the Redis key:
#   redis-eats:chunk:<chunk_id>
# ---------------------------------------------------------------------------

def prepare_records(chunks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Convert internal chunk dicts to the flat format RedisVL expects.
    The 'id' field is used as the unique key suffix.
    """
    records = []
    for chunk in chunks:
        records.append({
            "id":          chunk["chunk_id"],
            "text":        chunk["text"],
            "source":      chunk["source"],
            "page_number": chunk["page_number"],
            "chunk_index": chunk["chunk_index"],
            "embedding":   chunk["embedding"],  # RedisVL serializes this as FLOAT32 bytes
        })
    return records


records = prepare_records(all_chunks)
print(f"Prepared {len(records)} records for loading")

In [ ]:
# ---------------------------------------------------------------------------
# Load all records into Redis using RedisVL's batch loader
#
# index.load() writes all records in one efficient operation.
# Redis stores each as a Hash, and the vector index is updated automatically.
# ---------------------------------------------------------------------------

keys = index.load(records, id_field="id")

print(f"✅ Loaded {len(keys)} chunks into Redis Cloud")
print(f"   Sample key: {keys[0]}")

# Verify the count via index info
info = index.info()
print(f"\n   Index reports {info.get('num_docs', '?')} documents indexed")

### ✅ Checkpoint — Data is in Redis

Your chunks, metadata, and embedding vectors are now stored in Redis Cloud. You can verify this in **Redis Insight** by browsing keys matching `redis-eats:chunk:*`.

Run the cell below to spot-check one record directly.


In [ ]:
# ---------------------------------------------------------------------------
# Spot-check: fetch one chunk directly from Redis
# ---------------------------------------------------------------------------
sample_key = keys[0]
raw = r.hgetall(sample_key)

print(f"Key     : {sample_key}")
print(f"source  : {raw.get('source')}")
print(f"page    : {raw.get('page_number')}")
print(f"text    : {raw.get('text', '')[:200]}...")
print(f"embedding bytes: {len(raw.get('embedding', b''))} bytes "
      f"(= {len(raw.get('embedding', b'')) // 4} float32 values)")

---
## Section 7 — Vector Search

Now comes the core of RAG: **finding the chunks most relevant to a user's question**.

The process is:
1. Embed the user's question with the same model used to embed the chunks
2. Ask Redis to find the `k` chunks whose embeddings are closest (by cosine similarity)
3. Return those chunks as context for the LLM

Redis returns a **similarity score** with each result. For COSINE distance, a score of `0.0` means identical, `1.0` means completely unrelated — so lower scores = more relevant.


In [ ]:
# ---------------------------------------------------------------------------
# Vector search function
#
# Embeds the query, runs VectorQuery against Redis, returns top-k results.
# ---------------------------------------------------------------------------

def search_chunks(
    query: str,
    top_k: int = 5,
) -> List[Dict[str, Any]]:
    """
    Find the top_k most semantically similar chunks for the given query.

    Args:
        query: The user's question or search string.
        top_k: Number of results to return.

    Returns:
        List of result dicts with text, source, page_number, and score.
    """
    # Embed the query using the same model used for the chunks
    query_vector = get_embedding(query)

    # Build a RedisVL VectorQuery
    q = VectorQuery(
        vector=query_vector,
        vector_field_name="embedding",
        return_fields=["text", "source", "page_number", "chunk_index"],
        num_results=top_k,
    )

    # Execute the search
    results = index.query(q)

    return results


# Test search
results = search_chunks("Can I get a refund if my food arrived cold?", top_k=3)

print(f"Top 3 results for 'Can I get a refund if my food arrived cold?'\n")
for i, r_item in enumerate(results, 1):
    score = float(r_item.get('vector_distance', 0))
    print(f"  [{i}] source={r_item['source']}  page={r_item['page_number']}  score={score:.4f}")
    print(f"      {r_item['text'][:200]}...")
    print()

In [ ]:
# ---------------------------------------------------------------------------
# Try a few more example searches to see retrieval in action
# ---------------------------------------------------------------------------
example_queries = [
    "What happens if my delivery is late?",
    "How do promo codes work?",
    "How do I reset my account password?",
]

for query in example_queries:
    results = search_chunks(query, top_k=1)
    if results:
        best = results[0]
        score = float(best.get('vector_distance', 0))
        print(f"Q: {query}")
        print(f"   → {best['source']} (score={score:.4f})")
        print(f"   → {best['text'][:150]}...")
        print()

### ✅ Checkpoint — Try Your Own Question

Edit the query in the cell below and run it. Try questions about food safety, restaurant onboarding, driver procedures, or cancellations.


In [ ]:
# ---------------------------------------------------------------------------
# YOUR TURN — try your own food-delivery question
# ---------------------------------------------------------------------------
my_question = "What should a restaurant do if they need to pause orders?"

my_results = search_chunks(my_question, top_k=3)
print(f"Results for: '{my_question}'\n")
for i, r_item in enumerate(my_results, 1):
    score = float(r_item.get('vector_distance', 0))
    print(f"[{i}] {r_item['source']}  page={r_item['page_number']}  score={score:.4f}")
    print(f"    {r_item['text'][:200]}...")
    print()

---
## Section 8 — Build the RAG Answer Function

### How RAG Works

Retrieval-Augmented Generation combines search and generation:

```
1. User asks a question
2. We embed the question and search Redis for relevant chunks
3. We build a prompt: system message + retrieved chunks + user question
4. We send the prompt to the LLM
5. The LLM answers using only the provided context (no hallucination from training data)
6. We return the answer + citations so the user knows where it came from
```

The key insight: Redis **grounds** the LLM. Without retrieval, the LLM would guess. With retrieval, it answers from your actual policy documents.


In [ ]:
# ---------------------------------------------------------------------------
# Build a RAG prompt from retrieved chunks
# ---------------------------------------------------------------------------

SYSTEM_PROMPT = """You are Don't Talk With Food In Your Mouth, the Redis Eats customer support assistant.
Answer the customer's question using ONLY the policy information provided in the context below.
If the context does not contain enough information to answer the question, say so honestly.
Do not make up information or use knowledge outside of the provided context.
Be concise, friendly, and helpful."""


def build_prompt(question: str, retrieved_chunks: List[Dict[str, Any]]) -> str:
    """
    Build the user-turn content for the LLM from retrieved chunks and the question.

    Format:
        CONTEXT:
        [Source: <filename>, Page: <n>]
        <chunk text>
        ...

        QUESTION:
        <user question>

    Args:
        question: The user's question string.
        retrieved_chunks: List of chunk dicts returned by search_chunks().

    Returns:
        A formatted string to use as the user message content.
    """
    context_parts = []
    for chunk in retrieved_chunks:
        source = chunk.get("source", "unknown")
        page   = chunk.get("page_number", "?")
        text   = chunk.get("text", "")
        context_parts.append(f"[Source: {source}, Page: {page}]\n{text}")

    context_str = "\n\n".join(context_parts)

    return f"CONTEXT:\n{context_str}\n\nQUESTION:\n{question}"


# Test the prompt builder
test_results = search_chunks("Can I get a refund if my food arrived cold?", top_k=3)
test_prompt = build_prompt("Can I get a refund if my food arrived cold?", test_results)
print(test_prompt[:800], "...")

In [ ]:
# ---------------------------------------------------------------------------
# Core RAG answer function
#
# Retrieves relevant chunks, builds prompt, calls OpenAI, returns answer + citations.
# ---------------------------------------------------------------------------

CHAT_MODEL = "gpt-4o-mini"   # Affordable, fast, and capable for this workshop


def ask_rag(
    question: str,
    top_k: int = 5,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Answer a question using the Redis Eats RAG pipeline.

    Steps:
      1. Search Redis for the top_k most relevant chunks
      2. Build a grounded prompt
      3. Call OpenAI for an answer
      4. Return answer + source citations

    Args:
        question: The user's question.
        top_k:    Number of chunks to retrieve.
        verbose:  If True, print the result to the console.

    Returns:
        Dict with 'answer' and 'citations' keys.
    """
    # Step 1: Retrieve relevant chunks from Redis
    chunks = search_chunks(question, top_k=top_k)

    # Step 2: Build the grounded prompt
    user_message = build_prompt(question, chunks)

    # Step 3: Call the OpenAI chat model
    response = openai_client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_message},
        ],
        temperature=0.2,  # Low temperature for factual, consistent answers
    )
    answer = response.choices[0].message.content

    # Step 4: Compile citations from retrieved chunks
    seen = set()
    citations = []
    for chunk in chunks:
        citation = f"{chunk['source']} (page {chunk['page_number']})"
        if citation not in seen:
            citations.append(citation)
            seen.add(citation)

    if verbose:
        print(f"Question : {question}")
        print(f"\nAnswer   :\n{answer}")
        print(f"\nSources  : {', '.join(citations)}")

    return {"answer": answer, "citations": citations}


# Test the RAG pipeline
result = ask_rag("Can I get a refund if my food arrived cold?")

In [ ]:
# ---------------------------------------------------------------------------
# Run the full set of workshop example questions
# ---------------------------------------------------------------------------
workshop_questions = [
    "What happens if my delivery is late?",
    "How do promo codes work?",
    "What should a restaurant do if they need to pause orders?",
    "How do I reset my account password?",
]

for q in workshop_questions:
    print("=" * 60)
    ask_rag(q)
    print()

---
## Section 9 — Semantic Routing

### Why Route Before Retrieval?

Without a guard, users could ask our chatbot anything:

> *"Write me a poem about databases."*  
> *"Who won the Super Bowl?"*  
> *"How do I invest in stocks?"*

The RAG pipeline would dutifully embed those questions, search Redis, find the least-irrelevant chunks, and spend money calling OpenAI — all to produce a confused or hallucinated answer.

**Semantic routing** intercepts each question *before* retrieval and decides: *Is this question in-domain?* If not, return a refusal immediately — no embedding, no search, no LLM call.

### RedisVL SemanticRouter

RedisVL's `SemanticRouter` stores route definitions in Redis and uses vector similarity to match incoming questions. Routes are defined by **utterances** — example phrases that represent that category.

Our routes:

| Route | Covers |
|---|---|
| `food_delivery_support` | Refunds, delays, missing items, order issues |
| `account_and_login` | Password reset, account settings, payment methods |
| `restaurant_procedures` | Menu management, hours, onboarding, pausing orders |
| `driver_procedures` | Driver pay, delivery issues, safety, pickups |
| `out_of_domain` | Anything not food-delivery related → refused |


In [ ]:
# ---------------------------------------------------------------------------
# Import RedisVL SemanticRouter components
# ---------------------------------------------------------------------------
from redisvl.extensions.router import SemanticRouter, Route

print("✅ SemanticRouter imported")

In [ ]:
# ---------------------------------------------------------------------------
# Define semantic routes
#
# Each Route has a name and a list of utterances — representative phrases
# that describe questions belonging to that route.
#
# More diverse utterances = better coverage of different phrasings.
# ---------------------------------------------------------------------------

# Route 1: food delivery support and policy questions
route_food_delivery = Route(
    name="food_delivery_support",
    utterances=[
        "Can I get a refund for my order?",
        "My food arrived cold, what can I do?",
        "My order never arrived.",
        "What happens if my delivery is late?",
        "I want to cancel my order.",
        "My delivery is taking too long.",
        "Wrong items were delivered to me.",
        "How do I report a missing item?",
        "How do promo codes work?",
        "My promo code is not working.",
        "Is my food safe to eat?",
        "The food was damaged when it arrived.",
    ],
)

# Route 2: account and login help
route_account = Route(
    name="account_and_login",
    utterances=[
        "How do I reset my password?",
        "I can't log in to my account.",
        "How do I update my email address?",
        "How do I add a new payment method?",
        "I want to delete my account.",
        "How do I enable two-factor authentication?",
        "My account has been locked.",
        "How do I change my delivery address?",
    ],
)

# Route 3: restaurant and merchant procedures
route_restaurant = Route(
    name="restaurant_procedures",
    utterances=[
        "How do I pause orders on Redis Eats?",
        "How do I update my restaurant menu?",
        "How do I change my restaurant hours?",
        "How does restaurant onboarding work?",
        "When do restaurants get paid?",
        "How do I become a Redis Eats restaurant partner?",
        "What commission does Redis Eats charge restaurants?",
        "How do I manage my menu on the platform?",
    ],
)

# Route 4: driver and delivery procedures
route_driver = Route(
    name="driver_procedures",
    utterances=[
        "How do I report a problem during a delivery?",
        "What do I do if the restaurant isn't ready?",
        "How does driver pay work?",
        "How do I appeal a driver account suspension?",
        "What do I do if the customer doesn't answer?",
        "How do I contact driver support?",
        "Is my delivery partner account active?",
        "How do I report a safety incident as a driver?",
    ],
)

print("✅ Routes defined")
print(f"   food_delivery_support : {len(route_food_delivery.utterances)} utterances")
print(f"   account_and_login     : {len(route_account.utterances)} utterances")
print(f"   restaurant_procedures : {len(route_restaurant.utterances)} utterances")
print(f"   driver_procedures     : {len(route_driver.utterances)} utterances")

### 9.1 — Create the SemanticRouter

RedisVL stores the route embeddings in Redis and performs vector similarity when a new question arrives. The router reuses the same Redis connection as our search index — Redis is the backend for both.

The **threshold** controls how confident the router must be before assigning a route. We'll explore tuning this in the checkpoint below.


In [ ]:
# ---------------------------------------------------------------------------
# Create the SemanticRouter
#
# The router embeds each utterance and stores them in Redis.
# At query time it embeds the incoming question and finds the closest route.
#
# routing_threshold: cosine distance threshold (0.0–1.0)
#   Lower = stricter (fewer matches, more refusals)
#   Higher = looser  (more matches, some false positives)
# ---------------------------------------------------------------------------

ROUTING_THRESHOLD = 0.5   # Starting point — we'll tune this below

router = SemanticRouter(
    name="redis-eats-router",
    routes=[route_food_delivery, route_account, route_restaurant, route_driver],
    routing_threshold=ROUTING_THRESHOLD,
    redis_url=REDIS_URL,
    overwrite=True,   # Drop and recreate if already exists from a prior run
)

print(f"✅ SemanticRouter 'redis-eats-router' created")
print(f"   Routes           : {[r.name for r in router.routes]}")
print(f"   Routing threshold: {ROUTING_THRESHOLD}")

### 9.2 — Test the Router

Let's see the router classify both allowed and blocked questions.

- **Matched route** → question is in-domain, proceed to RAG
- **No match (None)** → question is out-of-domain, refuse immediately


In [ ]:
# ---------------------------------------------------------------------------
# Test the router on a set of allowed and blocked questions
# ---------------------------------------------------------------------------

allowed_questions = [
    "Can I get a refund if my food arrived cold?",
    "What happens if my delivery is late?",
    "How do I reset my account password?",
    "What should a restaurant do if they need to pause orders?",
    "How does driver pay work?",
]

blocked_questions = [
    "Who won the Super Bowl?",
    "What is the weather in Chicago?",
    "Write me a poem about databases.",
    "How do I invest in stocks?",
    "What is the capital of France?",
]

REFUSAL_MESSAGE = "I can't answer that question. I'm a food delivery bot."

print("--- ALLOWED QUESTIONS ---")
for q in allowed_questions:
    route_match = router(q)          # Returns a RouteMatch or None
    name = route_match.name if route_match else None
    status = f"✅ routed → {name}" if name else "⚠️  no match (consider lowering threshold)"
    print(f"  {status}")
    print(f"  Q: {q}")
    print()

print("--- BLOCKED QUESTIONS ---")
for q in blocked_questions:
    route_match = router(q)
    name = route_match.name if route_match else None
    if name:
        status = f"⚠️  incorrectly matched → {name} (consider raising threshold)"
    else:
        status = f"✅ blocked → refusal sent"
    print(f"  {status}")
    print(f"  Q: {q}")
    print()

### 9.3 — Threshold Tuning

The `routing_threshold` controls the distance cutoff. Try different values below.

| Threshold | Effect |
|---|---|
| `0.3` | Very strict — only very close matches are routed |
| `0.5` | Balanced — good default for this workshop |
| `0.7` | Lenient — more questions pass through, some false positives |

> **Workshop default:** `0.5`


In [ ]:
# ---------------------------------------------------------------------------
# Threshold tuning — change ROUTING_THRESHOLD and re-run this cell
# ---------------------------------------------------------------------------

ROUTING_THRESHOLD = 0.5   # Try 0.3, 0.5, 0.7 and observe the difference

# Update the router's threshold without recreating it
router.routing_threshold = ROUTING_THRESHOLD

# Quick test on an edge-case question
test_questions = [
    "My order is messed up",          # Vague but in-domain
    "Tell me something interesting",   # Vague and out-of-domain
    "help",                            # Very short — should be out-of-domain
]

print(f"Threshold = {ROUTING_THRESHOLD}")
print()
for q in test_questions:
    match = router(q)
    result = match.name if match else "→ REFUSED"
    print(f"  '{q}'")
    print(f"   {result}\n")

### 9.4 — RAG With Routing

Now we combine routing with the RAG pipeline from Section 8. Every question passes through the router first. Out-of-domain questions never reach Redis or OpenAI.


In [ ]:
# ---------------------------------------------------------------------------
# RAG pipeline with semantic routing guard
# ---------------------------------------------------------------------------

def ask_with_routing(
    question: str,
    top_k: int = 5,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Answer a question using semantic routing + RAG.

    Flow:
      1. Route the question via SemanticRouter
      2. If out-of-domain → refuse immediately (no Redis search, no LLM call)
      3. If in-domain     → run the full RAG pipeline from Section 8

    Args:
        question: The user's question.
        top_k:    Number of chunks to retrieve if in-domain.
        verbose:  If True, print the result.

    Returns:
        Dict with 'answer', 'citations', and 'route' keys.
    """
    # Step 1: Route the question
    route_match = router(question)
    route_name  = route_match.name if route_match else None

    # Step 2: Refuse if out-of-domain
    if route_name is None:
        if verbose:
            print(f"Question : {question}")
            print(f"Route    : [out-of-domain — refused]")
            print(f"\nAnswer   : {REFUSAL_MESSAGE}")
        return {"answer": REFUSAL_MESSAGE, "citations": [], "route": None}

    # Step 3: In-domain — run the RAG pipeline
    if verbose:
        print(f"Question : {question}")
        print(f"Route    : {route_name}")

    result = ask_rag(question, top_k=top_k, verbose=verbose)
    result["route"] = route_name
    return result


# Test: in-domain question
print("=" * 60)
ask_with_routing("Can I get a refund if my food arrived cold?")
print()

# Test: out-of-domain question
print("=" * 60)
ask_with_routing("Who won the Super Bowl?")

### ✅ Checkpoint — Routing in Action

Run the cell below and try both an allowed and a blocked question. Notice that blocked questions get an immediate refusal — **no embedding, no Redis search, no OpenAI call**.


In [ ]:
# ---------------------------------------------------------------------------
# YOUR TURN — try an allowed and a blocked question
# ---------------------------------------------------------------------------

print("=" * 60)
ask_with_routing("How do promo codes work?")          # in-domain
print()
print("=" * 60)
ask_with_routing("What is the capital of France?")    # out-of-domain

---
## Section 10 — LangCache: Semantic Caching

### The Problem: Repeated LLM Calls

Imagine a food delivery app where thousands of customers open the support chat every morning. Many of them type something like:

- `good morning`
- `Good morning!`
- `hey, good morning`
- `morning`

These are semantically identical — but without caching, each one triggers a full round-trip: embed → route → search Redis → call OpenAI. A Redis customer reported **meaningful cost savings** after caching just this type of repeated low-value input.

### What Is LangCache?

**Redis LangCache** is a fully-managed semantic caching service built on Redis Cloud. Instead of rolling your own vector cache (which takes code, tuning, and maintenance), LangCache gives you a simple API:

```
1. Search the cache: is there a similar prompt already answered?
2a. Cache HIT  → return the cached response instantly (no LLM call)
2b. Cache MISS → call the LLM, then store the response in the cache
```

The similarity check is semantic, not exact — so `"good morning"` and `"Good morning!"` both hit the same cache entry.

> **Note:** LangCache is currently in **preview** on Redis Cloud. Features and endpoints may evolve. See the [LangCache docs](https://redis.io/docs/latest/develop/ai/langcache/) for the latest.


In [ ]:
# ---------------------------------------------------------------------------
# LangCache credentials
#
# You need a LangCache instance provisioned on Redis Cloud.
# Find your URL, Cache ID, and API key in the Redis Cloud console.
#
# If you do not have a LangCache instance yet, you can still follow along —
# the cell below will skip gracefully if credentials are not provided.
# ---------------------------------------------------------------------------

LANGCACHE_URL      = input("LangCache URL (https://...): ").strip()
LANGCACHE_CACHE_ID = input("LangCache Cache ID: ").strip()
LANGCACHE_API_KEY  = getpass.getpass("LangCache API Key: ")

LANGCACHE_AVAILABLE = bool(LANGCACHE_URL and LANGCACHE_CACHE_ID and LANGCACHE_API_KEY)

if LANGCACHE_AVAILABLE:
    print("✅ LangCache credentials collected")
else:
    print("⚠️  LangCache credentials not provided — skipping live demo.")
    print("   You can still follow the code and understand how LangCache works.")

In [ ]:
# ---------------------------------------------------------------------------
# Initialise LangCache client
# ---------------------------------------------------------------------------
lang_cache = None

if LANGCACHE_AVAILABLE:
    try:
        from langcache import LangCache
        lang_cache = LangCache(
            server_url=LANGCACHE_URL,
            cache_id=LANGCACHE_CACHE_ID,
            api_key=LANGCACHE_API_KEY,
        )
        print("✅ LangCache client initialised")
    except Exception as e:
        print(f"❌ LangCache init failed: {e}")
        LANGCACHE_AVAILABLE = False

### 10.1 — Cache Miss and Cache Hit

We'll use the `"good morning"` family of inputs to demonstrate semantic caching.

**First call:** cache miss — LangCache has no entry yet, so we call the LLM and store the response.  
**Subsequent calls:** cache hit — semantically similar prompts return the cached answer instantly.

The similarity threshold controls how similar two prompts must be to share a cache entry. We start at `0.9` (very similar) — a good default for production semantic caching.


In [ ]:
# ---------------------------------------------------------------------------
# Helper: call LLM for a simple greeting response
# (This is a standalone demo — not part of the RAG pipeline)
# ---------------------------------------------------------------------------
def call_llm_for_greeting(prompt: str) -> str:
    """
    Generate a short greeting response from the LLM.
    Used only for the LangCache demonstration.

    Args:
        prompt: The user's greeting text.

    Returns:
        A friendly response string.
    """
    response = openai_client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": "You are a friendly food delivery support assistant. "
                                           "Reply to greetings warmly and briefly."},
            {"role": "user",   "content": prompt},
        ],
        temperature=0.7,
        max_tokens=60,
    )
    return response.choices[0].message.content


# ---------------------------------------------------------------------------
# LangCache wrapper: search → hit or miss → store
# ---------------------------------------------------------------------------
def ask_with_cache(prompt: str, similarity_threshold: float = 0.9) -> str:
    """
    Answer a prompt using LangCache semantic caching.

    Flow:
      1. Search LangCache for a semantically similar cached response
      2a. Cache HIT  → return cached response (no LLM call)
      2b. Cache MISS → call LLM, store response in cache, return response

    Args:
        prompt:               The user's input.
        similarity_threshold: How similar two prompts must be to share a cache entry.
                              0.9 = very similar (recommended); 0.8 = looser matching.

    Returns:
        The response string (from cache or freshly generated).
    """
    if not LANGCACHE_AVAILABLE or lang_cache is None:
        print(f"[LangCache unavailable — calling LLM directly]")
        return call_llm_for_greeting(prompt)

    # Step 1: Search the cache
    cached = lang_cache.search(
        prompt=prompt,
        similarity_threshold=similarity_threshold,
    )

    if cached:
        # Cache HIT — return immediately without calling the LLM
        response = cached[0]["response"]
        print(f"  📦 CACHE HIT  | '{prompt}'")
        print(f"     Response   : {response}")
        return response

    # Cache MISS — call the LLM and store the result
    print(f"  🔄 CACHE MISS | '{prompt}' — calling LLM...")
    response = call_llm_for_greeting(prompt)

    # Store in LangCache for future semantically-similar queries
    lang_cache.set(prompt=prompt, response=response)

    print(f"     Response   : {response}")
    print(f"     Stored in cache ✅")
    return response


print("✅ LangCache helper functions defined")

### 10.2 — Demonstration: 'Good Morning' Variants

Watch what happens as we send the same semantic intent with different wording. The first call will be a cache miss (LLM call). Subsequent similar inputs should be cache hits.


In [ ]:
# ---------------------------------------------------------------------------
# Good morning demo — run each variant and observe cache behaviour
# ---------------------------------------------------------------------------

import time

greeting_variants = [
    "good morning",
    "Good morning!",
    "hey, good morning",
    "morning",
]

CACHE_THRESHOLD = 0.85   # Tune this to see more or fewer cache hits

print(f"Similarity threshold: {CACHE_THRESHOLD}")
print("=" * 55)

for variant in greeting_variants:
    start = time.time()
    ask_with_cache(variant, similarity_threshold=CACHE_THRESHOLD)
    elapsed = time.time() - start
    print(f"     Latency    : {elapsed:.2f}s")
    print()

### 10.3 — Why LangCache vs Roll Your Own?

You *could* build semantic caching yourself using Redis vector search directly. But LangCache gives you:

| Feature | Roll Your Own | LangCache |
|---|---|---|
| Setup | Schema + index + code | One SDK call |
| Embedding | You manage the model | Managed by Redis Cloud |
| TTL / expiry | You implement | Configurable in the console |
| Monitoring | You build | Built-in cache hit metrics |
| Updates | You maintain | Managed service |

For customer-facing AI applications, LangCache is the recommended path — less code to maintain, fewer edge cases to handle, and it stays in Redis Cloud alongside your vector index and router.


---
## Section 11 — Final Chatbot Test

Time to bring it all together. The full pipeline is:

```
User question
  → SemanticRouter (RedisVL)        ← refuse if out-of-domain
  → LangCache search                ← return cached answer if hit
  → Vector search (Redis Cloud)     ← retrieve relevant policy chunks
  → OpenAI chat model               ← generate grounded answer
  → Answer + Citations              ← with source document references
  → LangCache store                 ← cache the response for next time
```

This is the **Don't Talk With Food In Your Mouth** Redis Eats support bot.


In [ ]:
# ---------------------------------------------------------------------------
# Full chatbot function — routing + caching + RAG
#
# This is the production-shaped version that combines all three layers:
#   1. SemanticRouter  — guard against out-of-domain questions
#   2. LangCache       — return cached answers for repeated questions
#   3. RAG pipeline    — retrieve from Redis + generate via OpenAI
# ---------------------------------------------------------------------------

def ask_bot(
    question: str,
    top_k: int = 5,
    cache_threshold: float = 0.9,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Don't Talk With Food In Your Mouth — Redis Eats Support Bot.

    Full pipeline:
      1. Route    : reject out-of-domain questions immediately
      2. Cache    : return cached responses for similar recent questions
      3. Retrieve : find relevant policy chunks via Redis vector search
      4. Generate : produce a grounded answer via OpenAI
      5. Store    : cache the new response in LangCache

    Args:
        question:        The user's question.
        top_k:           Number of chunks to retrieve.
        cache_threshold: Semantic similarity threshold for cache lookup.
        verbose:         Print results to the console.

    Returns:
        Dict with 'answer', 'citations', 'route', and 'cache_hit' keys.
    """
    if verbose:
        print(f"\n🤖 Don't Talk With Food In Your Mouth")
        print(f"   Q: {question}")
        print()

    # -----------------------------------------------------------------------
    # Step 1: Semantic routing
    # -----------------------------------------------------------------------
    route_match = router(question)
    route_name  = route_match.name if route_match else None

    if route_name is None:
        # Out-of-domain — refuse without touching Redis search or OpenAI
        if verbose:
            print(f"   [router] out-of-domain → refused")
            print(f"\n   A: {REFUSAL_MESSAGE}")
        return {
            "answer":    REFUSAL_MESSAGE,
            "citations": [],
            "route":     None,
            "cache_hit": False,
        }

    if verbose:
        print(f"   [router] route → {route_name}")

    # -----------------------------------------------------------------------
    # Step 2: LangCache lookup
    # -----------------------------------------------------------------------
    if LANGCACHE_AVAILABLE and lang_cache is not None:
        cached = lang_cache.search(
            prompt=question,
            similarity_threshold=cache_threshold,
        )
        if cached:
            cached_answer = cached[0]["response"]
            if verbose:
                print(f"   [cache]  HIT — returning cached response")
                print(f"\n   A: {cached_answer}")
                print(f"\n   Sources: (from cache)")
            return {
                "answer":    cached_answer,
                "citations": [],
                "route":     route_name,
                "cache_hit": True,
            }
        if verbose:
            print(f"   [cache]  MISS — proceeding to retrieval")

    # -----------------------------------------------------------------------
    # Step 3: RAG pipeline
    # -----------------------------------------------------------------------
    rag_result = ask_rag(question, top_k=top_k, verbose=verbose)

    # -----------------------------------------------------------------------
    # Step 4: Store in LangCache for next time
    # -----------------------------------------------------------------------
    if LANGCACHE_AVAILABLE and lang_cache is not None:
        lang_cache.set(prompt=question, response=rag_result["answer"])
        if verbose:
            print(f"   [cache]  response stored for future similar questions")

    rag_result["route"]     = route_name
    rag_result["cache_hit"] = False
    return rag_result


print("✅ ask_bot() is ready")

### 11.1 — Successful RAG Questions

These should all be routed correctly, miss the cache (first run), retrieve from Redis, and return a grounded answer with citations.


In [ ]:
# ---------------------------------------------------------------------------
# Run all workshop success-path example questions
# ---------------------------------------------------------------------------
success_questions = [
    "Can I get a refund if my food arrived cold?",
    "What happens if my delivery is late?",
    "How do promo codes work?",
    "What should a restaurant do if they need to pause orders?",
    "How do I reset my account password?",
]

for q in success_questions:
    ask_bot(q)
    print("-" * 60)

### 11.2 — Out-of-Domain Refusals

These should all be intercepted by the router and refused without any Redis search or LLM call.


In [ ]:
# ---------------------------------------------------------------------------
# Run all workshop out-of-domain examples
# ---------------------------------------------------------------------------
blocked = [
    "Who won the Super Bowl?",
    "What is the weather in Chicago?",
    "Write me a poem about databases.",
    "How do I invest in stocks?",
    "What is the capital of France?",
]

for q in blocked:
    ask_bot(q)
    print("-" * 60)

### 11.3 — LangCache Demonstration

Run this cell twice (or run the cell, then change the wording slightly and run again). The second pass should show cache hits for semantically similar inputs.


In [ ]:
# ---------------------------------------------------------------------------
# Ask the same question twice — observe cache miss then cache hit
# ---------------------------------------------------------------------------
repeat_question = "Can I get a refund if my food arrived cold?"

print("First call (expect MISS):")
ask_bot(repeat_question)

print("\nSecond call — semantically similar phrasing (expect HIT):")
ask_bot("My food was cold when it arrived. Am I entitled to a refund?")

---
## Section 12 — Reset Lab

Great work! Before you close out, let's clean up everything this workshop wrote to your Redis Cloud database.

This cell will:

1. **Drop the vector search index** (`redis-eats-chunks`) — removes the index definition
2. **Delete all chunk keys** matching `redis-eats:chunk:*` — removes the stored documents and vectors
3. **Drop the semantic router index** from Redis
4. **Confirm** the cleanup was successful

> ⚠️ **Safe by design:** The cleanup is scoped to the key prefix `redis-eats:` and the named indexes created during this workshop. It will not touch any other data in your database.

> 💡 **Why does this matter?** When using a shared or persistent database, always clean up workshop data so it doesn't accumulate, consume memory, or interfere with future work.


In [ ]:
# ---------------------------------------------------------------------------
# Reset Lab — clean up all workshop data from Redis Cloud
#
# This is scoped to:
#   - Index   : redis-eats-chunks
#   - Keys    : redis-eats:chunk:*
#   - Router  : redis-eats-router (managed by RedisVL)
# ---------------------------------------------------------------------------

import time

print("Starting Redis Eats workshop cleanup...\n")

# -----------------------------------------------------------------------
# Step 1: Drop the chunk vector search index
# SearchIndex.delete() drops the index AND deletes all indexed keys
# when delete_documents=True.
# -----------------------------------------------------------------------
try:
    index.delete(drop=True)   # drop=True removes the index definition
    print("✅ Vector search index 'redis-eats-chunks' dropped")
except Exception as e:
    print(f"⚠️  Index drop: {e}")

# -----------------------------------------------------------------------
# Step 2: Delete any remaining redis-eats:chunk:* keys
# (belt-and-suspenders — index.delete() should handle this,
#  but we scan to confirm zero keys remain)
# -----------------------------------------------------------------------
try:
    chunk_keys = list(r.scan_iter("redis-eats:chunk:*", count=500))
    if chunk_keys:
        r.delete(*chunk_keys)
        print(f"✅ Deleted {len(chunk_keys)} remaining chunk keys")
    else:
        print("✅ No stray chunk keys found (index.delete() cleaned them up)")
except Exception as e:
    print(f"⚠️  Chunk key cleanup: {e}")

# -----------------------------------------------------------------------
# Step 3: Delete the semantic router from Redis
# -----------------------------------------------------------------------
try:
    router.delete()   # Removes the router index and its stored embeddings
    print("✅ SemanticRouter 'redis-eats-router' deleted")
except Exception as e:
    print(f"⚠️  Router cleanup: {e}")

# -----------------------------------------------------------------------
# Step 4: Verification — confirm nothing remains
# -----------------------------------------------------------------------
time.sleep(0.5)   # Brief pause to let Redis process the deletes

remaining_keys = list(r.scan_iter("redis-eats:*", count=500))
if remaining_keys:
    print(f"\n⚠️  {len(remaining_keys)} redis-eats:* keys still present:")
    for k in remaining_keys[:10]:
        print(f"   {k}")
else:
    print("\n✅ All redis-eats:* keys removed — database is clean")

print("\n🏁 Reset complete. Your Redis Cloud database is back to its pre-workshop state.")

### Verify in Redis Insight (Optional)

Open **Redis Insight** and browse your database. You should see zero keys matching `redis-eats:*` and no indexes named `redis-eats-chunks` or `redis-eats-router`.

Run the cell below for a quick in-notebook confirmation.


In [ ]:
# ---------------------------------------------------------------------------
# Final verification — count remaining workshop keys
# ---------------------------------------------------------------------------
count = sum(1 for _ in r.scan_iter("redis-eats:*", count=500))
print(f"Keys matching 'redis-eats:*' : {count}")

# Check indexes
try:
    indexes = r.execute_command("FT._LIST")
    workshop_indexes = [idx for idx in indexes if b"redis-eats" in idx or "redis-eats" in str(idx)]
    if workshop_indexes:
        print(f"Workshop indexes still present: {workshop_indexes}")
    else:
        print("Workshop indexes             : none (clean)")
except Exception:
    print("(Could not list indexes — this is fine if the index module is not available)")

---
## Section 13 — What Comes Next

Congratulations — you built a working RAG chatbot on Redis Cloud! 🎉

Here's what you accomplished in this workshop:

| ✅ | Skill |
|---|---|
| ✅ | Connected a Jupyter Notebook to Redis Cloud |
| ✅ | Loaded and chunked PDF policy documents |
| ✅ | Generated embeddings with OpenAI |
| ✅ | Created a RedisVL vector search index |
| ✅ | Stored documents, metadata, and vectors together in Redis |
| ✅ | Ran pure vector search over Redis Query Engine |
| ✅ | Built a grounded RAG answer function with citations |
| ✅ | Added semantic routing to block off-topic questions |
| ✅ | Added LangCache to reduce repeated LLM calls |

---

### Workshop 2: Context-Aware Redis Eats

The next workshop builds on this foundation and turns the chatbot into a **context-aware, action-capable agent**:

| Feature | Workshop 1 | Workshop 2 |
|---|---|---|
| Policy Q&A (RAG) | ✅ | ✅ |
| Semantic routing | ✅ | ✅ |
| LangCache | ✅ | ✅ |
| Conversational memory | ❌ | ✅ |
| Live order lookup | ❌ | ✅ |
| Delivery status | ❌ | ✅ |
| MCP tools (Context Retriever) | ❌ | ✅ |
| Redis Iris integration | ❌ | ✅ |
| Agentic workflows | ❌ | ✅ |
| RDI (mocked) | ❌ | ✅ |

---

### Keep Exploring Redis AI

- 📖 [RedisVL documentation](https://www.redisvl.com)
- 📖 [Redis Vector Search guide](https://redis.io/docs/latest/develop/interact/search-and-query/advanced-concepts/vectors/)
- 📖 [LangCache documentation](https://redis.io/docs/latest/develop/ai/langcache/)
- 📖 [Redis RAG Quickstart](https://redis.io/docs/latest/develop/get-started/rag/)
- 🚀 [Redis Cloud free tier](https://redis.io/try-free)

---

### Ideas to Try on Your Own

- Swap in **your own PDF documents** and change the system prompt
- Add **hybrid search** — combine vector similarity with a tag filter (e.g., only search `refund_policy.pdf`)
- Increase `top_k` and compare answer quality
- Adjust `chunk_size` and measure whether retrieval improves
- Try **HNSW** instead of FLAT for the vector index and measure speed on a larger dataset

Thanks for attending the Redis Eats RAG Workshop. Now go build something fast. 🚀
